# RTU11 hybrid FDD: rule-based Open-FDD + ML residual (NF sidecar example)

Same idea as the upstream [RTU7_machine_learning.ipynb](https://github.com/bbartling/open-fdd/blob/main/examples/AHU/RTU7_machine_learning.ipynb) in **open-fdd**:

1. **Rules** — YAML `RuleRunner` (bounds, flatline, cookbook-style expression) on Brick-named columns.
2. **ML** — Regressor predicts **Supply_Air_Temperature_Sensor** from other available signals when the fan is on; **residual** vs prediction; flag rows above the **95th percentile** of absolute error (tunable).

**Why both?** Rules give explicit, auditable limits; ML catches *expectation vs reality* drift before it always breaches a fixed threshold.

**Data:** `RTU11.csv` + `rtu11_column_map.yaml` (this repo). **No Normal Framework** required — the production NF sidecar swaps the CSV step for HPL `/api/v1/point/data`.

Start Jupyter from `solutions/open-fdd-rules-engine` (repo root of the sidecar package).

## Install (once)

```bash
cd solutions/open-fdd-rules-engine
python -m venv .venv && source .venv/bin/activate
pip install -r examples/AHU/requirements-ml.txt
python -m ipykernel install --user --name=openfdd-nf-ml --display-name="OpenFDD NF ML"
```

Then select that kernel in this notebook.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from open_fdd.engine import RuleRunner
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.tree import DecisionTreeRegressor

from openfdd_nf.offline_csv import dataframe_from_mapped_csv, load_column_map

# Run Jupyter with cwd = solutions/open-fdd-rules-engine
ROOT = Path.cwd().resolve()
if not (ROOT / "openfdd_nf").is_dir():
    raise RuntimeError(
        "Start Jupyter from solutions/open-fdd-rules-engine (openfdd_nf package missing)."
    )

EXAMPLES = ROOT / "examples" / "AHU"
CSV_PATH = EXAMPLES / "RTU11.csv"
MAP_PATH = EXAMPLES / "rtu11_column_map.yaml"
RULES_DIR = EXAMPLES / "rules_demo"

plt.style.use("seaborn-v0_8-whitegrid")

# --- ML / analysis knobs ---
TARGET = "Supply_Air_Temperature_Sensor"
FAN_COL = "Supply_Fan_Speed_Command"
FAN_MIN = 0.01  # fan "on" in 0–1 space after column map scale
ML_FAULT_QUANTILE = 0.95
ML_PRED_COL = "sat_pred_ml"
ML_RESID_COL = "sat_residual_ml"
ML_ABS_RESID_COL = "sat_abs_residual_ml"
ML_FAULT_FLAG = "ml_residual_fault"

In [ ]:
spec = load_column_map(MAP_PATH)
df = dataframe_from_mapped_csv(CSV_PATH, spec)
print(df.shape, df["timestamp"].min(), df["timestamp"].max())
df.head()

In [ ]:
runner = RuleRunner(rules_path=RULES_DIR)
df_result = runner.run(df, timestamp_col="timestamp", skip_missing_columns=True)
flag_cols = [c for c in df_result.columns if c.endswith("_flag")]
print("Rule flag columns:", flag_cols)
for c in flag_cols:
    s = df_result[c].fillna(0).astype(bool)
    print(f"  {c}: true_count={int(s.sum())} last={int(bool(s.iloc[-1])) if len(s) else 0}")

In [ ]:
# Fan-on mask (RTU11 has no separate fan status; use command)
df_result = df_result.copy()
df_result["fan_on"] = (df_result[FAN_COL].fillna(0) > FAN_MIN).astype(bool)
print("fan_on rows:", int(df_result["fan_on"].sum()), "/", len(df_result))

# Features: only columns present in this CSV (narrower than full AHU7 notebook)
LAG_SOURCES = [
    "Outside_Air_Temperature_Sensor",
    "Return_Air_Temperature_Sensor",
    "Mixed_Air_Temperature_Sensor",
]
for col in LAG_SOURCES:
    if col in df_result.columns:
        df_result[f"{col}_lag1"] = df_result[col].shift(1)

BASE_FEATURES = [
    "Outside_Air_Temperature_Sensor",
    "Return_Air_Temperature_Sensor",
    "Mixed_Air_Temperature_Sensor",
    "Damper_Position_Command",
    "Supply_Fan_Speed_Command",
    "Supply_Air_Static_Pressure_Sensor",
]
FEATURE_COLS = [
    c for c in BASE_FEATURES if c in df_result.columns and c != TARGET
] + [f"{c}_lag1" for c in LAG_SOURCES if f"{c}_lag1" in df_result.columns]

print("Using features:", FEATURE_COLS)

model_df = df_result.loc[df_result["fan_on"]].copy()
need = [TARGET] + FEATURE_COLS
model_df = model_df.dropna(subset=need)
print("ML training rows:", len(model_df))

if len(model_df) < 80:
    raise ValueError("Not enough fan-on rows after dropna for a stable ML demo.")

X = model_df[FEATURE_COLS]
y = model_df[TARGET]

In [ ]:
tscv = TimeSeriesSplit(n_splits=min(4, max(2, len(model_df) // 200)))
print("TimeSeriesSplit get_n_splits:", tscv.get_n_splits())

param_grid = [
    {
        "estimator": [DecisionTreeRegressor(random_state=42)],
        "estimator__max_depth": [4, 8, 12, None],
        "estimator__min_samples_leaf": [2, 5, 10],
    },
    {
        "estimator": [RandomForestRegressor(random_state=42, n_jobs=-1)],
        "estimator__n_estimators": [100, 200],
        "estimator__max_depth": [8, 16, None],
    },
    {
        "estimator": [ExtraTreesRegressor(random_state=42, n_jobs=-1)],
        "estimator__n_estimators": [100, 200],
        "estimator__max_depth": [8, 16, None],
    },
]

from sklearn.pipeline import Pipeline

pipe = Pipeline([("estimator", RandomForestRegressor())])

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    refit=True,
)
grid.fit(X, y)
print("Best MAE (neg):", grid.best_score_)
print("Best params:", grid.best_params_)

y_pred = grid.predict(X)
model_df = model_df.assign(**{ML_PRED_COL: y_pred})
model_df[ML_RESID_COL] = model_df[TARGET] - model_df[ML_PRED_COL]
model_df[ML_ABS_RESID_COL] = model_df[ML_RESID_COL].abs()

thresh = model_df[ML_ABS_RESID_COL].quantile(ML_FAULT_QUANTILE)
print(f"Residual threshold ({ML_FAULT_QUANTILE:.0%} quantile): {thresh:.3f} °F equiv.")
model_df[ML_FAULT_FLAG] = (model_df[ML_ABS_RESID_COL] >= thresh).astype(int)

# Merge ML columns back to full timeline (NaN where not trained / fan off)
for col in [ML_PRED_COL, ML_RESID_COL, ML_ABS_RESID_COL, ML_FAULT_FLAG]:
    df_result[col] = np.nan
df_result.loc[model_df.index, ML_PRED_COL] = model_df[ML_PRED_COL].values
df_result.loc[model_df.index, ML_RESID_COL] = model_df[ML_RESID_COL].values
df_result.loc[model_df.index, ML_ABS_RESID_COL] = model_df[ML_ABS_RESID_COL].values
df_result.loc[model_df.index, ML_FAULT_FLAG] = model_df[ML_FAULT_FLAG].values

In [ ]:
# Compare rule vs ML fault density on fan-on rows
sub = df_result.loc[df_result["fan_on"]].dropna(subset=[TARGET])
rule_any = sub[flag_cols].fillna(0).astype(bool).any(axis=1) if flag_cols else pd.Series(False, index=sub.index)
ml_f = sub[ML_FAULT_FLAG].fillna(0).astype(bool) if ML_FAULT_FLAG in sub.columns else pd.Series(False, index=sub.index)
print("Fan-on rows:", len(sub))
print("Any rule flag:", int(rule_any.sum()))
print("ML residual flag:", int(ml_f.sum()))
print("Both:", int((rule_any & ml_f).sum()))
print("Rule only:", int((rule_any & ~ml_f).sum()))
print("ML only:", int((~rule_any & ml_f).sum()))

In [ ]:
def shade_from_flag(ax, times, flag_series, color, alpha=0.22):
    flag_series = flag_series.fillna(0).astype(bool)
    in_run = False
    start = None
    for i, (t, on) in enumerate(zip(times, flag_series)):
        if on and not in_run:
            in_run = True
            start = t
        elif not on and in_run:
            ax.axvspan(start, times.iloc[i - 1], color=color, alpha=alpha)
            in_run = False
    if in_run:
        ax.axvspan(start, times.iloc[-1], color=color, alpha=alpha)


def _flag_series(df, col):
    return df[col] if col in df.columns else pd.Series(0, index=df.index)


plot_df = df_result.dropna(subset=[TARGET]).iloc[:2000]  # first N rows for clarity
t = plot_df["timestamp"]
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(t, plot_df[TARGET], label="SAT (actual)", color="black", lw=1)
if plot_df[ML_PRED_COL].notna().any():
    axes[0].plot(t, plot_df[ML_PRED_COL], label="SAT (ML pred)", color="tab:blue", alpha=0.75)
shade_from_flag(axes[0], t, _flag_series(plot_df, "bad_sensor_flag"), "tab:red")
shade_from_flag(axes[0], t, _flag_series(plot_df, ML_FAULT_FLAG), "tab:orange")
axes[0].set_ylabel("°F")
axes[0].legend(loc="upper right")
axes[0].set_title("Supply air temp: rule bounds (red) vs ML residual fault (orange)")

axes[1].plot(t, plot_df[ML_RESID_COL], color="tab:green", lw=0.8, label="residual")
if plot_df[ML_ABS_RESID_COL].notna().any():
    thr = plot_df[ML_ABS_RESID_COL].quantile(ML_FAULT_QUANTILE)
    axes[1].axhline(thr, color="tab:orange", ls="--", label=f"{ML_FAULT_QUANTILE:.0%} abs residual")
    axes[1].axhline(-thr, color="tab:orange", ls="--")
axes[1].set_ylabel("residual °F")
axes[1].legend(loc="upper right")
plt.tight_layout()
plt.show()

## Next steps

- Tighten **ML_FAULT_QUANTILE** (e.g. 0.99) for fewer ML flags, or loosen to 0.90.
- Add **AHU7.csv** and a column map to reproduce the richer feature set from [RTU7_machine_learning.ipynb](https://github.com/bbartling/open-fdd/blob/main/examples/AHU/RTU7_machine_learning.ipynb).
- Wire the same **Brick columns** into the NF sidecar **`mapping.yaml`** and call **`POST /run`** for rules-only production; keep ML in a notebook or batch job that reads exported HPL CSV.